# From slide to result

Aurora predicts spatial gene expression from routine H&E images, helping
researchers explore molecular patterns across samples, cohorts and disease.
This notebook takes one H&E slide to its result: one `.h5ad` of predicted
spatial gene expression, on your slide's own coordinates.
[02-downstream.ipynb](02-downstream.ipynb) explores it.

**Choose the workflow that fits your research.** Both return the same file.

- **Send your slides.** Prepare your H&E slide on your own machine and send its
  tiles.
- **Your slides stay with you.** Run DeepSpot-H, the foundation model for H&E
  images, on your own machine and send the numbers it produces. No image leaves
  your machine.

DeepSpot-M, the model that predicts spatial gene expression, runs on ours in
both.

Running this notebook end to end needs a key.
[Get access](https://docs.auroraomics.org/get-access/) says how to ask for one.

## 1. Choose your workflow

`ROUTE` is the one setting: `"patches"` to send your slides, or `"embeddings"`
to keep them with you.

In [ ]:
import sys
from pathlib import Path

import anndata
import numpy as np

import auroraomics as ao
from auroraomics.client import Client

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from quickstart import zenodo  # noqa: E402

ROUTE = "patches"  # "patches": send your slides. "embeddings": your slides stay with you.
assert ROUTE in ("patches", "embeddings"), ROUTE

DATA = ROOT / "data"
OBSERVATIONS = DATA / ("sample.zip" if ROUTE == "patches" else "sample.npz")
PREDICTION = DATA / "prediction.h5ad"

print("auroraomics", ao.__version__, "| route:", ROUTE)

## 2. Fetch the example slide

One lung cancer section, `LC1`, from an open-access dataset:

> Dawo, S., Nonchev, K., & Silina, K. (2025). *10x Visium Spatial
> Transcriptomics Dataset: Kidney (3) and Lung (5) Cancer with Tertiary
> Lymphoid Structures* [Data set]. Zenodo.
> <https://doi.org/10.5281/zenodo.14620362>

`quickstart/zenodo.py` fetches that one slide from the record, not the whole
archive. To use your own slide, set `slide_path` to its file.

In [ ]:
slide_path = zenodo.fetch("slide", DATA, on_progress=zenodo.progress)
print(slide_path.name)

## 3. Open the slide

`open_slide` reads the slide's resolution from the file, and refuses a file
that records none. If yours records none, pass it:
`ao.open_slide(path, mpp=...)`. Only the part being read is decoded, so a slide
larger than memory works.

In [ ]:
slide = ao.open_slide(slide_path)
print(slide)

## 4. Ask the service what the model needs

The quality standard every tile is checked against comes from the service, so
your machine and the service agree on it. The model's card says what the
model reads, and reading it needs your key: run
`auroraomics login you@institution.edu` once on this machine.

In [ ]:
client = Client()
thresholds = client.qc_thresholds()

card = next(c for c in client.models() if ROUTE in c["accepts"]["observations"])
print("model:", card["id"])
print(f"genes: {card['output']['n_genes']:,}")

## 5. Send your slides

`pack_tiles` cuts the slide into tiles, checks each one and writes the ones
that pass into one archive. It reports what it dropped and why.
[Check tile quality](https://docs.auroraomics.org/check-tiles/) says what each
reason means.

In [ ]:
if ROUTE == "patches":
    report = ao.pack_tiles(
        slide,
        OBSERVATIONS,
        mpp=slide.mpp,
        patch_um=card["input_spec"]["patch_um"],
        thresholds=thresholds,
    )
    print(f"{report.written:,} tiles kept of {report.considered:,}")
    print("dropped by:", report.rejected_by_reason or "nothing")

## 6. Your slides stay with you

DeepSpot-H runs on your machine, and no image is uploaded. It needs one extra:

```bash
pip install "auroraomics[embed]"
```

On a machine with no GPU, ask for the CPU build as well:

```bash
pip install "auroraomics[embed]" --extra-index-url https://download.pytorch.org/whl/cpu
```

It runs on a CPU, and uses a GPU when there is one. Your key lets the package
ask the service which release of the weights to use, so every machine computes
the same numbers. The weights download once, onto your machine.

In [ ]:
if ROUTE == "embeddings":
    report = ao.embed_tiles(
        slide,
        OBSERVATIONS,
        mpp=slide.mpp,
        patch_um=card["input_spec"]["patch_um"],
        encoder=ao.resolve_encoder(client.encoders()),
        thresholds=thresholds,
    )
    print(f"{report.rows:,} tiles embedded of {report.considered:,}")
    print("dropped by:", report.rejected_by_reason or "nothing")

The file holds, for each tile, a row of numbers, the tile's position on the
slide and the quality measurements that kept it. There is no tile, no
thumbnail, no slide label and no macro image in it. Open it and check:

In [ ]:
if ROUTE == "embeddings":
    with np.load(OBSERVATIONS, allow_pickle=False) as bundle:
        print(sorted(bundle.files))

## 7. Submit and wait

The submission is the same on both workflows. The file is uploaded in parts,
so an interrupted upload resumes, and `wait` polls at the interval the service
asks for.

To explore the result as a map in
[Aurora Intelligence](https://app.auroraomics.org), set
`land_in_workspace=True` and import it into a workspace.

In [ ]:
job = client.predict(
    model=card["id"],
    observations={ROUTE: OBSERVATIONS},
    land_in_workspace=False,
)
print(job.id, job.state)

job = job.wait(on_poll=lambda j: print(j.state, "| queue position:", j.queue_position))
job.save(PREDICTION)
print("saved", PREDICTION.name)

## 8. What you get back

One `.h5ad`, the same on both workflows: one row per tile, one column per gene,
and each tile's centre on your slide under `obsm["spatial"]`. Genes are indexed
by Ensembl identifier, and the symbol sits beside it in `var["feature_name"]`.
Keep the file: the copy on the service expires.
[What you get back](https://docs.auroraomics.org/results/) is the full layout.

In [ ]:
adata = anndata.read_h5ad(PREDICTION)
print(adata)
print(adata.var[["feature_name", "measured_in_training"]].head())

## Next

[02-downstream.ipynb](02-downstream.ipynb) explores this file on your tissue:
where a gene's signal appears, a UMAP of the predicted expression, and clusters
drawn back on the slide.